# Customer Segmentation — Unsupervised Clustering Models

Exploring and comparing multiple unsupervised clustering algorithms on the Mall Customers dataset:

| Model | Type |
|---|---|
| K-Means | Centroid-based |
| DBSCAN | Density-based |
| Agglomerative Clustering | Hierarchy-based |
| Mean Shift | Density-based (auto k) |
| Gaussian Mixture Model | Probabilistic |


## 1. Load & Explore the Dataset


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler

df = pd.read_csv('Mall_Customers.csv')

print('Shape:', df.shape)
print()
print('Missing values:')
print(df.isnull().sum())
print()
df.head(10)


In [ ]:
# Statistical summary
df.describe()


In [ ]:
# Feature distributions
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle('Feature Distributions', fontsize=13, fontweight='bold')

sns.histplot(df['Age'], kde=True, ax=axes[0], color='steelblue')
axes[0].set_title('Age')

sns.histplot(df['Annual Income (k$)'], kde=True, ax=axes[1], color='coral')
axes[1].set_title('Annual Income (k$)')

sns.histplot(df['Spending Score (1-100)'], kde=True, ax=axes[2], color='forestgreen')
axes[2].set_title('Spending Score (1-100)')

plt.tight_layout()
plt.show()

# Gender split
print('Gender split:')
gender_col = 'Gender' if 'Gender' in df.columns else 'Genre'
print(df[gender_col].value_counts())


## 2. Feature Selection & Scaling
We use **Annual Income** and **Spending Score** — the two most discriminative features.


In [ ]:
X = df[['Annual Income (k$)', 'Spending Score (1-100)']].values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print('Feature matrix shape:', X_scaled.shape)


## 3. K-Means Clustering


In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# Elbow method
wcss = []
sil_scores = []
K_range = range(2, 11)

for k in K_range:
    km = KMeans(n_clusters=k, init='k-means++', random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    wcss.append(km.inertia_)
    sil_scores.append(silhouette_score(X_scaled, labels))

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(K_range, wcss, marker='o', linestyle='--', color='steelblue')
axes[0].set_title('Elbow Method — WCSS')
axes[0].set_xlabel('Number of Clusters (k)')
axes[0].set_ylabel('WCSS')

axes[1].plot(K_range, sil_scores, marker='o', linestyle='--', color='coral')
axes[1].set_title('Silhouette Score vs k')
axes[1].set_xlabel('Number of Clusters (k)')
axes[1].set_ylabel('Silhouette Score')

plt.tight_layout()
plt.show()

best_k = K_range[sil_scores.index(max(sil_scores))]
print(f'Best k by silhouette: {best_k}  (score={max(sil_scores):.4f})')


In [ ]:
# Fit final KMeans with k=5
km_final = KMeans(n_clusters=5, init='k-means++', random_state=42, n_init=10)
km_labels = km_final.fit_predict(X_scaled)

plt.figure(figsize=(8, 6))
colors = ['red', 'blue', 'green', 'cyan', 'magenta']
for i in range(5):
    plt.scatter(X[km_labels == i, 0], X[km_labels == i, 1],
                s=80, c=colors[i], label=f'Cluster {i+1}', alpha=0.7)
plt.scatter(km_final.cluster_centers_[:, 0] * scaler.scale_[0] + scaler.mean_[0],
            km_final.cluster_centers_[:, 1] * scaler.scale_[1] + scaler.mean_[1],
            s=300, c='yellow', marker='X', label='Centroids', zorder=5)
plt.title('K-Means Clustering (k=5)')
plt.xlabel('Annual Income (k$)')
plt.ylabel('Spending Score (1-100)')
plt.legend()
plt.tight_layout()
plt.show()

km_sil = silhouette_score(X_scaled, km_labels)
print(f'K-Means Silhouette Score: {km_sil:.4f}')


## 4. DBSCAN — Density-Based Clustering
DBSCAN finds clusters of arbitrary shape and automatically labels noise points as **-1**.
Key parameters:
- `eps` — neighbourhood radius
- `min_samples` — minimum points to form a dense region


In [ ]:
from sklearn.cluster import DBSCAN

# Tune eps using k-distance graph
from sklearn.neighbors import NearestNeighbors

nbrs = NearestNeighbors(n_neighbors=5).fit(X_scaled)
distances, _ = nbrs.kneighbors(X_scaled)
distances = np.sort(distances[:, 4])[::-1]

plt.figure(figsize=(8, 4))
plt.plot(distances, color='steelblue')
plt.title('K-Distance Graph (k=5) — find elbow for eps')
plt.xlabel('Points sorted by distance')
plt.ylabel('5th Nearest Neighbour Distance')
plt.axhline(y=0.5, color='red', linestyle='--', label='eps=0.5')
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# Fit DBSCAN
db = DBSCAN(eps=0.5, min_samples=5)
db_labels = db.fit_predict(X_scaled)

n_clusters = len(set(db_labels)) - (1 if -1 in db_labels else 0)
n_noise    = (db_labels == -1).sum()

print(f'Clusters found : {n_clusters}')
print(f'Noise points   : {n_noise}')

plt.figure(figsize=(8, 6))
unique_labels = set(db_labels)
palette = plt.cm.tab10(np.linspace(0, 1, len(unique_labels)))

for label, color in zip(unique_labels, palette):
    if label == -1:
        plt.scatter(X[db_labels == label, 0], X[db_labels == label, 1],
                    s=60, c='black', marker='x', label='Noise', alpha=0.6)
    else:
        plt.scatter(X[db_labels == label, 0], X[db_labels == label, 1],
                    s=80, color=color, label=f'Cluster {label+1}', alpha=0.7)

plt.title(f'DBSCAN Clustering (eps=0.5, min_samples=5) — {n_clusters} clusters, {n_noise} noise')
plt.xlabel('Annual Income (k$)')
plt.ylabel('Spending Score (1-100)')
plt.legend()
plt.tight_layout()
plt.show()

if n_clusters > 1:
    mask = db_labels != -1
    db_sil = silhouette_score(X_scaled[mask], db_labels[mask])
    print(f'DBSCAN Silhouette Score (excl. noise): {db_sil:.4f}')


## 5. Agglomerative (Hierarchical) Clustering
Builds a tree of clusters bottom-up. The **dendrogram** shows where to cut.


In [ ]:
from sklearn.cluster import AgglomerativeClustering
from scipy.cluster.hierarchy import dendrogram, linkage

# Dendrogram on a sample for readability
linked = linkage(X_scaled, method='ward')

plt.figure(figsize=(14, 5))
dendrogram(linked, truncate_mode='lastp', p=20,
           leaf_rotation=45, leaf_font_size=10)
plt.title('Dendrogram (Ward linkage) — truncated to last 20 merges')
plt.xlabel('Cluster size')
plt.ylabel('Distance')
plt.tight_layout()
plt.show()


In [ ]:
# Fit Agglomerative with k=5
agg = AgglomerativeClustering(n_clusters=5, linkage='ward')
agg_labels = agg.fit_predict(X_scaled)

plt.figure(figsize=(8, 6))
colors = ['red', 'blue', 'green', 'cyan', 'magenta']
for i in range(5):
    plt.scatter(X[agg_labels == i, 0], X[agg_labels == i, 1],
                s=80, c=colors[i], label=f'Cluster {i+1}', alpha=0.7)
plt.title('Agglomerative Clustering (k=5, Ward linkage)')
plt.xlabel('Annual Income (k$)')
plt.ylabel('Spending Score (1-100)')
plt.legend()
plt.tight_layout()
plt.show()

agg_sil = silhouette_score(X_scaled, agg_labels)
print(f'Agglomerative Silhouette Score: {agg_sil:.4f}')


## 6. Mean Shift Clustering
Mean Shift **automatically determines the number of clusters** by finding density peaks. No k needed.


In [ ]:
from sklearn.cluster import MeanShift, estimate_bandwidth

bandwidth = estimate_bandwidth(X_scaled, quantile=0.2, n_samples=200)
ms = MeanShift(bandwidth=bandwidth, bin_seeding=True)
ms_labels = ms.fit_predict(X_scaled)

n_ms_clusters = len(np.unique(ms_labels))
print(f'Bandwidth used   : {bandwidth:.4f}')
print(f'Clusters found   : {n_ms_clusters}')

plt.figure(figsize=(8, 6))
palette = plt.cm.tab10(np.linspace(0, 1, n_ms_clusters))
for i, color in zip(np.unique(ms_labels), palette):
    plt.scatter(X[ms_labels == i, 0], X[ms_labels == i, 1],
                s=80, color=color, label=f'Cluster {i+1}', alpha=0.7)
plt.scatter(ms.cluster_centers_[:, 0] * scaler.scale_[0] + scaler.mean_[0],
            ms.cluster_centers_[:, 1] * scaler.scale_[1] + scaler.mean_[1],
            s=300, c='yellow', marker='X', label='Centroids', zorder=5)
plt.title(f'Mean Shift Clustering ({n_ms_clusters} clusters auto-detected)')
plt.xlabel('Annual Income (k$)')
plt.ylabel('Spending Score (1-100)')
plt.legend()
plt.tight_layout()
plt.show()

if n_ms_clusters > 1:
    ms_sil = silhouette_score(X_scaled, ms_labels)
    print(f'Mean Shift Silhouette Score: {ms_sil:.4f}')


## 7. Gaussian Mixture Model (GMM)
GMM is a **probabilistic** model — each cluster is a Gaussian distribution. Unlike K-Means it assigns **soft probabilities** (a point can partially belong to multiple clusters).


In [ ]:
from sklearn.mixture import GaussianMixture

# BIC to find optimal number of components
bic_scores = []
aic_scores = []
n_range = range(2, 11)

for n in n_range:
    gmm = GaussianMixture(n_components=n, random_state=42)
    gmm.fit(X_scaled)
    bic_scores.append(gmm.bic(X_scaled))
    aic_scores.append(gmm.aic(X_scaled))

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(n_range, bic_scores, marker='o', label='BIC', color='steelblue')
ax.plot(n_range, aic_scores, marker='s', label='AIC', color='coral')
ax.set_title('GMM — BIC / AIC vs Number of Components')
ax.set_xlabel('Number of components')
ax.set_ylabel('Score (lower = better)')
ax.legend()
plt.tight_layout()
plt.show()

best_n = n_range[bic_scores.index(min(bic_scores))]
print(f'Optimal components by BIC: {best_n}')


In [ ]:
# Fit final GMM
gmm_final = GaussianMixture(n_components=5, random_state=42)
gmm_labels = gmm_final.fit_predict(X_scaled)
gmm_probs  = gmm_final.predict_proba(X_scaled).max(axis=1)  # confidence per point

plt.figure(figsize=(8, 6))
colors = ['red', 'blue', 'green', 'cyan', 'magenta']
for i in range(5):
    plt.scatter(X[gmm_labels == i, 0], X[gmm_labels == i, 1],
                s=80, c=colors[i], label=f'Component {i+1}', alpha=0.7)
plt.title('Gaussian Mixture Model (5 components)')
plt.xlabel('Annual Income (k$)')
plt.ylabel('Spending Score (1-100)')
plt.legend()
plt.tight_layout()
plt.show()

gmm_sil = silhouette_score(X_scaled, gmm_labels)
print(f'GMM Silhouette Score     : {gmm_sil:.4f}')
print(f'Mean assignment confidence: {gmm_probs.mean():.4f}')


## 8. Side-by-Side Comparison


In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(22, 5))
fig.suptitle('Clustering Algorithms — Side-by-Side', fontsize=14, fontweight='bold')

configs = [
    ('K-Means',        km_labels),
    ('DBSCAN',         db_labels),
    ('Agglomerative',  agg_labels),
    ('GMM',            gmm_labels),
]

for ax, (title, labels) in zip(axes, configs):
    unique = np.unique(labels)
    palette = plt.cm.tab10(np.linspace(0, 1, len(unique)))
    for lbl, color in zip(unique, palette):
        marker = 'x' if lbl == -1 else 'o'
        c = 'black' if lbl == -1 else color
        lname = 'Noise' if lbl == -1 else f'C{lbl+1}'
        ax.scatter(X[labels == lbl, 0], X[labels == lbl, 1],
                   s=40, c=[c], marker=marker, label=lname, alpha=0.7)
    ax.set_title(title)
    ax.set_xlabel('Annual Income (k$)')
    ax.set_ylabel('Spending Score')
    ax.legend(fontsize=7)

plt.tight_layout()
plt.show()


In [ ]:
# Silhouette score summary
results = {
    'K-Means'       : silhouette_score(X_scaled, km_labels),
    'Agglomerative' : silhouette_score(X_scaled, agg_labels),
    'GMM'           : silhouette_score(X_scaled, gmm_labels),
}
if n_clusters > 1:
    mask = db_labels != -1
    results['DBSCAN (no noise)'] = silhouette_score(X_scaled[mask], db_labels[mask])
if n_ms_clusters > 1:
    results['Mean Shift'] = silhouette_score(X_scaled, ms_labels)

summary = pd.DataFrame(results.items(), columns=['Model', 'Silhouette Score'])
summary = summary.sort_values('Silhouette Score', ascending=False).reset_index(drop=True)
print(summary.to_string(index=False))

# Bar chart
plt.figure(figsize=(9, 4))
plt.bar(summary['Model'], summary['Silhouette Score'], color='steelblue', edgecolor='white')
plt.title('Silhouette Score Comparison (higher = better separated clusters)')
plt.ylabel('Silhouette Score')
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()
